<a href="https://colab.research.google.com/github/Nayab-khalid/FlyRank-AI-Internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nayab-khalid/FlyRank-AI-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [1]:
# 1. Question
#
# Research Question:
# Can observable search, content and freshness signals help a content team rank
# pages so that limited review time can be focused on pages associated with an
# observed declining trend?
#
# Decision supported:
# Which content pages should be reviewed first.
#
# The model produces a ranking score. A human reviewer decides what action to
# take: refresh, protect and refresh, CTR review, engagement review, expand,
# or monitor.
#
# Cost of a wrong call:
# A false positive wastes editorial time. A false negative misses a page that
# deserved review. Acting on a recommendation without human review risks
# damaging a page that was working.
#
# The model is decision support. It does not change content automatically and
# it does not predict Google's ranking algorithm.

print("Lane: Refresh and Content Opportunity Scoring")
print("Task type: binary classification used for ranking")
print("Unit of analysis: one content page")
print("Output: a score per page, ranked highest first")
print("Decision: which pages a human reviews first")


Lane: Refresh and Content Opportunity Scoring
Task type: binary classification used for ranking
Unit of analysis: one content page
Output: a score per page, ranked highest first
Decision: which pages a human reviews first


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [2]:
# 2. Data
#
# Executed modeling work uses the FlyRank anonymized starter dataset.
# The larger warehouse release was used in w03 to verify the data contract,
# not to train this model.
#
# Excluded fields and why:
#   trend_direction       defines the target
#   trend_pct             derived from the same trend information
#   is_declining_label    the target itself
#   content_id            pseudonymous identifier
#   client_id             grouping key for validation only
#   the six 30-day window columns   see section 3, they rebuild the label
#
# No client names, private URLs, private domains or private search queries
# appear anywhere in this notebook.

import os

import numpy as np
import pandas as pd

possible_paths = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "/content/data/raw/content_refresh_anonymized.csv"
]

csv_path = next((p for p in possible_paths if os.path.exists(p)), None)

if csv_path is None:
    csv_path = (
        "https://raw.githubusercontent.com/"
        "Nayab-khalid/FlyRank-AI-Internship/"
        "main/data/raw/content_refresh_anonymized.csv"
    )

df = pd.read_csv(csv_path)

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Rows:", len(df))
print("Source columns:", df.shape[1] - 1)
print("Pseudonymized clients:", df["client_id"].nunique())

print("\nLabel distribution:")
display(
    df["is_declining_label"]
    .value_counts()
    .rename_axis("is_declining_label")
    .reset_index(name="rows")
)

print("Positive class base rate:", round(df["is_declining_label"].mean(), 4))


Rows: 30000
Source columns: 44
Pseudonymized clients: 32

Label distribution:


,is_declining_label,rows
0,1,16262
1,0,13738


Positive class base rate: 0.5421


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [3]:
# 3. Methodology
#
# Binary classification. Logistic Regression, because the target has two
# outcomes, the model returns a probability that can rank pages, and the
# coefficients can be read.
#
# The baseline was built first in w04: a transparent score combining staleness
# (180+ days since update) and visibility (500+ impressions in 90 days).
#
# Validation: 80/20 split grouped by client, seed 42, so no test client appears
# in training.
#
# LEAKAGE: my Week-3 and Week-6 audits checked forbidden columns BY NAME and
# both passed. They could not see that the label was still rebuildable from
# columns that are individually allowed. The check below is the one that found
# it.

numeric_features = [
    "impressions_90d", "clicks_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "days_with_impressions",
    "days_with_sessions", "impressions_last_30d", "clicks_last_30d",
    "sessions_last_30d", "impressions_prev_30d", "clicks_prev_30d",
    "sessions_prev_30d", "word_count", "char_count", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct"
]

categorical_features = ["content_type", "main_intent", "competition_level"]

print("Original feature count:",
      len(numeric_features) + len(categorical_features))

# ------------------------------------------------------------
# THE CHECK THAT FOUND THE LEAK
# ------------------------------------------------------------
#
# docs/data-dictionary.md defines the label as:
#   trend_pct       = (impressions_last_30d - impressions_prev_30d)
#                     / impressions_prev_30d * 100
#   trend_direction = "down" when trend_pct < -20
#
# Both ingredients were in my feature list.

prev_30d = pd.to_numeric(df["impressions_prev_30d"], errors="coerce")
last_30d = pd.to_numeric(df["impressions_last_30d"], errors="coerce")

documented_rule = (
    ((last_30d - prev_30d) / prev_30d.replace(0, np.nan)) * 100 < -20
).fillna(False).astype(int)

agreement = (documented_rule == df["is_declining_label"]).mean()

print("\nLEAKAGE CHECK")
print("Documented rule vs label agreement:", round(agreement, 6))
print("Positives predicted:", int(documented_rule.sum()),
      "| actual:", int(df["is_declining_label"].sum()))
print("Verdict:", "FAIL - label is rebuildable" if agreement > 0.99 else "PASS")

RECENT_WINDOW = [
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d"
]

# Keep the original list, in its original column order, so the "before"
# run below reproduces the published w05/w06 numbers exactly.
numeric_full = list(numeric_features)

numeric_features = [c for c in numeric_features if c not in RECENT_WINDOW]

print("\nRemoved:", RECENT_WINDOW)
print("Corrected feature count:",
      len(numeric_features) + len(categorical_features))


Original feature count: 26

LEAKAGE CHECK
Documented rule vs label agreement: 1.0
Positives predicted: 16262 | actual: 16262
Verdict: FAIL - label is rebuildable

Removed: ['impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d']
Corrected feature count: 20


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [4]:
# 4. Results (vs baseline)
#
# The model and the Week-4 baseline are scored on the same held-out test rows.
# Both feature sets are reported: the difference between them is the finding.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def make_model(numeric):
    pre = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore",
                                     sparse_output=False))
        ]), categorical_features)
    ])
    return Pipeline([
        ("preprocessor", pre),
        ("model", LogisticRegression(max_iter=1000,
                                     class_weight="balanced",
                                     random_state=42))
    ])


def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    order = np.argsort(-np.asarray(scores))[:k]
    return y_true[order].mean()


y = df["is_declining_label"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(
    splitter.split(df, y, groups=df["client_id"])
)

test_rows = df.iloc[test_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(train_idx), "| Test rows:", len(test_idx))
print("Training clients:", df.iloc[train_idx]["client_id"].nunique(),
      "| Test clients:", test_rows["client_id"].nunique())
print("Client overlap:", len(
    set(df.iloc[train_idx]["client_id"]) & set(test_rows["client_id"])
))
print("Test base rate:", round(y_test.mean(), 4))

# ------------------------------------------------------------
# WEEK-4 BASELINE ON THE SAME TEST ROWS
# ------------------------------------------------------------

imp = pd.to_numeric(test_rows["impressions_90d"], errors="coerce").fillna(0)
stale_days = pd.to_numeric(
    test_rows["days_since_last_update"], errors="coerce"
).fillna(0)

staleness = np.select([stale_days >= 180, stale_days >= 90], [2, 1], default=0)
visibility = np.select(
    [imp >= 3000, imp >= 500, imp >= 100], [3, 2, 1], default=0
)

baseline_score = pd.Series(
    staleness + visibility, index=test_rows.index
).astype(float)

baseline_score = baseline_score + imp.rank(method="average") / (
    len(test_rows) * 1000000
)

# ------------------------------------------------------------
# BOTH FEATURE SETS
# ------------------------------------------------------------

results = [{
    "method": "Week-4 baseline",
    "features": "-",
    "average_precision": average_precision_score(y_test, baseline_score),
    "roc_auc": roc_auc_score(y_test, baseline_score),
    "precision_at_50": precision_at_k(y_test, baseline_score)
}]

feature_sets = {
    "20 (corrected)": numeric_features,
    "26 (with leak)": numeric_full
}

scores = {}

for name, numeric in feature_sets.items():
    columns = numeric + categorical_features
    model = make_model(numeric)
    model.fit(df[columns].iloc[train_idx], y.iloc[train_idx])
    probability = model.predict_proba(df[columns].iloc[test_idx])[:, 1]
    scores[name] = probability
    results.append({
        "method": "Logistic Regression",
        "features": name,
        "average_precision": average_precision_score(y_test, probability),
        "roc_auc": roc_auc_score(y_test, probability),
        "precision_at_50": precision_at_k(y_test, probability)
    })

comparison = pd.DataFrame(results)

print("\nMODEL VS BASELINE (same held-out test rows)")
display(comparison)

# ------------------------------------------------------------
# ERROR ANALYSIS ON THE CORRECTED MODEL
# ------------------------------------------------------------

ranked = test_rows.assign(
    model_probability=scores["20 (corrected)"], actual=y_test
).sort_values("model_probability", ascending=False)

print("\nPrecision at depth, corrected model (test base rate "
      + str(round(y_test.mean(), 3)) + "):")

for k in (20, 50, 100, 500):
    print("  P@" + str(k).ljust(4), round(ranked.head(k)["actual"].mean(), 4))

top_false_positives = ranked[ranked["actual"] == 0].head(10)

print("\nTop 10 false positives:")
print("  distinct clients:", top_false_positives["client_id"].nunique())
print("  median impressions_90d:",
      top_false_positives["impressions_90d"].median())
print("  trend_direction mix:",
      dict(top_false_positives["trend_direction"].value_counts()))


Training rows: 23837 | Test rows: 6163
Training clients: 25 | Test clients: 7
Client overlap: 0
Test base rate: 0.511



MODEL VS BASELINE (same held-out test rows)


,method,features,average_precision,roc_auc,precision_at_50
0,Week-4 baseline,-,0.489069,0.506511,0.3
1,Logistic Regression,20 (corrected),0.596008,0.594877,0.7
2,Logistic Regression,26 (with leak),0.871535,0.849583,1.0



Precision at depth, corrected model (test base rate 0.511):
  P@20   0.8
  P@50   0.7
  P@100  0.67
  P@500  0.672

Top 10 false positives:
  distinct clients: 1
  median impressions_90d: 759.0
  trend_direction mix: {'up': np.int64(7), 'stable': np.int64(2), 'new': np.int64(1)}


## 5. Limitations

*What this work cannot claim.*

In [5]:
# 5. Limitations
#
# The target is an observed decline label, not a future outcome.
#
# The data is observational. This project cannot prove that refreshing a page
# causes its search performance to improve.
#
# The features are contemporaneous with the label. The surviving 90-day
# aggregates cover a window that contains the 30 days the label is measured
# over, so this is an association study and not a forecast.
#
# Seasonality, demand, competition, consolidation and cannibalization can all
# affect search performance and can resemble content decline.
#
# Low traffic volume makes percentage-based signals unstable.
#
# Client history is not evenly distributed, and Search Console and Analytics
# availability differ across clients and time periods.
#
# The corrected result is modest. ROC AUC near 0.59 is much closer to chance
# than to a dependable ranking.
#
# My first two leakage audits passed while the label was fully rebuildable.
# Audits that check column names cannot certify a feature set.
#
# The model was evaluated once under this design. There is no sealed holdout,
# so this is a validation estimate rather than a blind single-shot evaluation.

print("Claim language check")
print("-" * 40)

allowed = ["observed", "measured", "directional", "decision support"]
forbidden = ["causes", "guarantees", "predicts Google", "will recover"]

print("Allowed:", ", ".join(allowed))
print("Forbidden:", ", ".join(forbidden))

print("""
Final claim:

In this dataset, Logistic Regression showed measured and directional ability
to rank pages associated with the observed decline label. The output is
decision support for prioritizing human content review. It is not a prediction
of Google's ranking system and it does not establish why a page declined.
""")


Claim language check
----------------------------------------
Allowed: observed, measured, directional, decision support
Forbidden: causes, guarantees, predicts Google, will recover

Final claim:

In this dataset, Logistic Regression showed measured and directional ability
to rank pages associated with the observed decline label. The output is
decision support for prioritizing human content review. It is not a prediction
of Google's ranking system and it does not establish why a page declined.



## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [6]:
# 6. Ranked recommendations
#
# The action queue from w07, recomputed here so the numbers in the paper can be
# checked against a real run.
#
# The queue is ranked by a transparent observable-signal action score, not by
# the model probability. Reason codes explain every row.


def num(column):
    return pd.to_numeric(df[column], errors="coerce").fillna(0)


impressions = num("impressions_90d")
sessions = num("sessions_90d")
age = num("content_age_days")
freshness = num("days_since_last_update")
words = num("word_count")
position = num("avg_position")
page_ctr = num("ctr")
engagement = num("engagement_rate")
scroll = num("scroll_rate")

visibility_score = np.clip(
    np.log1p(impressions) / max(np.log1p(impressions.max()), 1) * 100, 0, 100
)
freshness_score = np.clip(freshness / 365 * 100, 0, 100)
position_score = np.where(
    position > 0, np.clip((30 - position) / 30 * 100, 0, 100), 0
)
depth_score = np.clip((1200 - words) / 1200 * 100, 0, 100)

action_score = (
    0.40 * visibility_score
    + 0.30 * freshness_score
    + 0.25 * position_score
    + 0.05 * depth_score
)

trend = df["trend_direction"].fillna("").astype(str).str.lower()


def reason_codes(i):
    reasons = []
    if freshness[i] >= 180 and impressions[i] >= 500:
        reasons.append("stale_visible_page")
    if trend[i] == "down" and impressions[i] >= 100:
        reasons.append("declining_with_demand")
    if 0 < words[i] < 1200 and impressions[i] >= 250:
        reasons.append("thin_visible_page")
    if 0 < position[i] <= 10 and age[i] >= 180:
        reasons.append("page_one_decay_risk")
    if impressions[i] >= 500 and 0 < position[i] <= 20 and page_ctr[i] < 0.5:
        reasons.append("low_ctr_visible_page")
    if sessions[i] >= 30 and (engagement[i] < 30 or scroll[i] < 30):
        reasons.append("low_engagement_visible_page")
    if not reasons:
        reasons.append("monitor")
    return "|".join(dict.fromkeys(reasons))


def action_from_reason(reason):
    reasons = set(reason.split("|"))
    if "page_one_decay_risk" in reasons:
        return "protect_and_refresh"
    if "stale_visible_page" in reasons or "declining_with_demand" in reasons:
        return "refresh"
    if "thin_visible_page" in reasons:
        return "expand"
    if "low_ctr_visible_page" in reasons:
        return "ctr_review"
    if "low_engagement_visible_page" in reasons:
        return "engagement_review"
    return "monitor"


codes = pd.Series([reason_codes(i) for i in range(len(df))])
recommended_action = codes.apply(action_from_reason)

score_80 = pd.Series(action_score).quantile(0.80)
score_50 = pd.Series(action_score).quantile(0.50)

confidence = np.where(
    (action_score >= score_80) & (impressions >= 500) & (sessions >= 10),
    "high",
    np.where(action_score >= score_50, "medium", "low")
)

print("Queue size:", len(df))

print("\nRecommended actions:")
display(recommended_action.value_counts().rename_axis(
    "recommended_action").reset_index(name="pages"))

print("Confidence levels:")
display(pd.Series(confidence).value_counts().rename_axis(
    "confidence").reset_index(name="pages"))

print("""
NO-GO LIST. The system must never automatically publish content, delete or
prune pages, merge URLs, change canonical decisions, rewrite metadata without
review, claim that an edit will cause recovery, or override an experienced
reviewer because a score is high.

The model ranks the work. A human decides what should happen.
""")


Queue size: 30000

Recommended actions:


,recommended_action,pages
0,refresh,10326
1,monitor,8976
2,protect_and_refresh,7076
3,ctr_review,2220
4,engagement_review,1374
5,expand,28


Confidence levels:


,confidence,pages
0,low,15000
1,medium,10229
2,high,4771



NO-GO LIST. The system must never automatically publish content, delete or
prune pages, merge URLs, change canonical decisions, rewrite metadata without
review, claim that an edit will cause recovery, or override an experienced
reviewer because a score is high.

The model ranks the work. A human decides what should happen.



## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [7]:
# 7. Artifacts the paper embeds
#
# Everything below is computed in this notebook, so the deployed paper can be
# checked against a real run.

print("=" * 62)
print("ARTIFACT 1 - MODEL VS BASELINE")
print("=" * 62)
display(comparison.round(6))

print("=" * 62)
print("ARTIFACT 2 - CONTENT ACTION QUEUE")
print("=" * 62)

queue_summary = (
    recommended_action.value_counts()
    .rename_axis("recommended_action")
    .reset_index(name="pages")
)
queue_summary["share"] = (
    queue_summary["pages"] / len(df) * 100
).round(1)
display(queue_summary)

print("=" * 62)
print("ARTIFACT 3 - THE LEAK, BEFORE AND AFTER")
print("=" * 62)

leak_effect = comparison.set_index("features")["average_precision"]

print("Baseline AP                    :", round(leak_effect["-"], 6))
print("LR with the leak (26 features) :",
      round(leak_effect["26 (with leak)"], 6))
print("LR corrected (20 features)     :",
      round(leak_effect["20 (corrected)"], 6))
print("\nApparent gain that was leakage :", round(
    leak_effect["26 (with leak)"] - leak_effect["20 (corrected)"], 6))
print("Real gain over the baseline    :", round(
    leak_effect["20 (corrected)"] - leak_effect["-"], 6))

print("\nDeployed paper: https://nayab-khalid.github.io/FlyRank-AI-Internship/")


ARTIFACT 1 - MODEL VS BASELINE


,method,features,average_precision,roc_auc,precision_at_50
0,Week-4 baseline,-,0.489069,0.506511,0.3
1,Logistic Regression,20 (corrected),0.596008,0.594877,0.7
2,Logistic Regression,26 (with leak),0.871535,0.849583,1.0


ARTIFACT 2 - CONTENT ACTION QUEUE


,recommended_action,pages,share
0,refresh,10326,34.4
1,monitor,8976,29.9
2,protect_and_refresh,7076,23.6
3,ctr_review,2220,7.4
4,engagement_review,1374,4.6
5,expand,28,0.1


ARTIFACT 3 - THE LEAK, BEFORE AND AFTER
Baseline AP                    : 0.489069
LR with the leak (26 features) : 0.871535
LR corrected (20 features)     : 0.596008

Apparent gain that was leakage : 0.275527
Real gain over the baseline    : 0.106939

Deployed paper: https://nayab-khalid.github.io/FlyRank-AI-Internship/


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
